# Azure Face API - Quickstart con SDK de Python

Este notebook demuestra cómo utilizar **Azure Cognitive Services Face API** con el SDK oficial de Python `azure-ai-vision-face`.

## Descripción del Servicio

Azure Face API es un servicio de inteligencia artificial que proporciona algoritmos avanzados para:

| Funcionalidad | Descripción |
|---------------|-------------|
| **Detección de rostros** | Identifica rostros humanos en imágenes con alta precisión |
| **Verificación facial** | Compara dos rostros para determinar si pertenecen a la misma persona |
| **Identificación facial** | Encuentra coincidencias de un rostro en un grupo de personas |
| **Análisis de calidad** | Evalúa si una imagen es apta para reconocimiento facial |
| **Large Person Group** | Administra grupos de personas para identificación a escala |

## Configuración del Recurso en Azure

| Parámetro | Valor |
|-----------|-------|
| **Endpoint** | Cargado desde `.env` |
| **Región** | East US 2 |
| **Modelo de Detección** | DETECTION_03 |
| **Modelo de Reconocimiento** | RECOGNITION_04 |

---

## Flujo del Notebook

1. **Instalación** del cliente SDK
2. **Configuración** de credenciales desde `.env`
3. **Creación** de Large Person Group
4. **Registro** de personas con sus imágenes faciales
5. **Entrenamiento** del modelo
6. **Identificación** de rostros en imágenes de prueba
7. **Verificación** de identidad
8. **Limpieza** de recursos

## 1. Instalación de la Librería Cliente

Instalamos el SDK oficial de Azure para Face API y `python-dotenv` para cargar variables de entorno.

In [ ]:
!pip install --upgrade azure-ai-vision-face python-dotenv

## 2. Importación de Módulos

Importamos todos los módulos necesarios del SDK de Azure Face API:

| Módulo | Propósito |
|--------|----------|
| `AzureKeyCredential` | Autenticación con clave API |
| `FaceAdministrationClient` | Administración de grupos de personas |
| `FaceClient` | Detección e identificación de rostros |
| `FaceAttributeTypeRecognition04` | Atributos faciales para modelo Recognition04 |
| `FaceDetectionModel` | Modelos de detección disponibles |
| `FaceRecognitionModel` | Modelos de reconocimiento disponibles |
| `QualityForRecognition` | Niveles de calidad de imagen |

In [ ]:
import os
import time
import uuid
from dotenv import load_dotenv

from azure.core.credentials import AzureKeyCredential
from azure.ai.vision.face import FaceAdministrationClient, FaceClient
from azure.ai.vision.face.models import (
    FaceAttributeTypeRecognition04,
    FaceDetectionModel,
    FaceRecognitionModel,
    QualityForRecognition
)

print("✅ Módulos importados correctamente")

## 3. Configuración de Credenciales

Cargamos las credenciales desde el archivo `.env` para mayor seguridad.

### Archivo `.env`

```
FACE_ENDPOINT=https://...
FACE_APIKEY=your-api-key
```

> **⚠️ Nota de Seguridad**: Nunca subas el archivo `.env` a repositorios públicos.

In [ ]:
# Cargar variables de entorno desde .env
load_dotenv()

# Credenciales desde variables de entorno
KEY = os.getenv("FACE_APIKEY")
ENDPOINT = os.getenv("FACE_ENDPOINT")

# ID único para el Large Person Group
LARGE_PERSON_GROUP_ID = str(uuid.uuid4())

print("Configuración cargada:")
print(f"  📍 Endpoint: {ENDPOINT}")
print(f"  🔑 Key: {KEY[:10]}...{KEY[-5:] if KEY else 'NOT SET'}")
print(f"  👥 Large Person Group ID: {LARGE_PERSON_GROUP_ID}")

## 4. Inicialización de Clientes

Creamos instancias autenticadas de los clientes de Azure Face API.

### Clientes Disponibles

| Cliente | Funcionalidad |
|---------|---------------|
| `FaceAdministrationClient` | Crear/eliminar grupos, agregar personas, entrenar modelos |
| `FaceClient` | Detectar rostros, identificar, verificar |

In [ ]:
face_admin_client = FaceAdministrationClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

face_client = FaceClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

print("✅ Clientes inicializados correctamente")

## 5. Creación del Large Person Group

Un **Large Person Group** es un contenedor que puede almacenar hasta 1 millón de personas, cada una con múltiples imágenes faciales.

### Características del Large Person Group

| Característica | Valor |
|----------------|-------|
| **Capacidad máxima** | 1,000,000 personas |
| **Rostros por persona** | Hasta 248 |
| **Modelo de reconocimiento** | RECOGNITION_04 (más preciso) |

In [ ]:
print(f"Creando Large Person Group: {LARGE_PERSON_GROUP_ID}")

face_admin_client.large_person_group.create(
    large_person_group_id=LARGE_PERSON_GROUP_ID,
    name=LARGE_PERSON_GROUP_ID,
    recognition_model=FaceRecognitionModel.RECOGNITION04,
)

print(f"✅ Large Person Group creado exitosamente")

## 6. Creación de Personas en el Grupo

Creamos tres personas dentro del grupo: **Woman**, **Man** y **Child**.

In [ ]:
# Crear persona: Woman
woman = face_admin_client.large_person_group.create_person(
    large_person_group_id=LARGE_PERSON_GROUP_ID,
    name="Woman",
)
print(f"✅ Persona 'Woman' creada con ID: {woman.person_id}")

# Crear persona: Man
man = face_admin_client.large_person_group.create_person(
    large_person_group_id=LARGE_PERSON_GROUP_ID,
    name="Man",
)
print(f"✅ Persona 'Man' creada con ID: {man.person_id}")

# Crear persona: Child
child = face_admin_client.large_person_group.create_person(
    large_person_group_id=LARGE_PERSON_GROUP_ID,
    name="Child",
)
print(f"✅ Persona 'Child' creada con ID: {child.person_id}")

## 7. Definición de Imágenes de Entrenamiento

Definimos las URLs de las imágenes que usaremos para entrenar el modelo.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/"

woman_images = [
    f"{BASE_URL}Family1-Mom1.jpg",
    f"{BASE_URL}Family1-Mom2.jpg",
]

man_images = [
    f"{BASE_URL}Family1-Dad1.jpg",
    f"{BASE_URL}Family1-Dad2.jpg",
]

child_images = [
    f"{BASE_URL}Family1-Son1.jpg",
    f"{BASE_URL}Family1-Son2.jpg",
]

print("📷 Imágenes de entrenamiento definidas:")
print(f"  Woman: {len(woman_images)} imágenes")
print(f"  Man: {len(man_images)} imágenes")
print(f"  Child: {len(child_images)} imágenes")

## 8. Función para Agregar Rostros con Validación de Calidad

In [ ]:
def add_faces_to_person(person, person_name, image_urls):
    """
    Agrega rostros de alta calidad a una persona en el Large Person Group.
    """
    faces_added = 0
    
    for image in image_urls:
        detected_faces = face_client.detect_from_url(
            url=image,
            detection_model=FaceDetectionModel.DETECTION03,
            recognition_model=FaceRecognitionModel.RECOGNITION04,
            return_face_id=True,
            return_face_attributes=[FaceAttributeTypeRecognition04.QUALITY_FOR_RECOGNITION],
        )
        
        if len(detected_faces) != 1:
            print(f"  ⚠️ Imagen omitida: {len(detected_faces)} rostros detectados")
            continue
        
        face = detected_faces[0]
        
        if face.face_attributes.quality_for_recognition != QualityForRecognition.HIGH:
            print(f"  ⚠️ Imagen omitida: calidad {face.face_attributes.quality_for_recognition}")
            continue
        
        face_admin_client.large_person_group.add_face_from_url(
            large_person_group_id=LARGE_PERSON_GROUP_ID,
            person_id=person.person_id,
            url=image,
            detection_model=FaceDetectionModel.DETECTION03,
        )
        
        print(f"  ✅ Face {face.face_id} agregado a {person_name}")
        faces_added += 1
    
    return faces_added

print("✅ Función add_faces_to_person definida")

## 9. Registro de Rostros para Cada Persona

In [ ]:
print("=" * 50)
print("REGISTRANDO ROSTROS")
print("=" * 50)

print("\n👩 Procesando imágenes de Woman:")
woman_faces = add_faces_to_person(woman, "Woman", woman_images)

print("\n👨 Procesando imágenes de Man:")
man_faces = add_faces_to_person(man, "Man", man_images)

print("\n👦 Procesando imágenes de Child:")
child_faces = add_faces_to_person(child, "Child", child_images)

print("\n" + "=" * 50)
print(f"RESUMEN: {woman_faces + man_faces + child_faces} rostros registrados en total")
print("=" * 50)

## 10. Entrenamiento del Large Person Group

In [ ]:
print(f"🏋️ Iniciando entrenamiento del grupo: {LARGE_PERSON_GROUP_ID}")

poller = face_admin_client.large_person_group.begin_train(
    large_person_group_id=LARGE_PERSON_GROUP_ID,
    polling_interval=5,
)

poller.wait()

print(f"✅ El grupo {LARGE_PERSON_GROUP_ID} ha sido entrenado exitosamente")

## 11. Imagen de Prueba para Identificación

In [ ]:
test_image = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/identification1.jpg"

print(f"📷 Imagen de prueba: {test_image}")
print("\n⏳ Pausando 60 segundos para evitar rate limiting...")

for i in range(60, 0, -10):
    print(f"   {i} segundos restantes...")
    time.sleep(10)

print("\n✅ Pausa completada")

## 12. Detección de Rostros en la Imagen de Prueba

In [ ]:
print("🔍 Detectando rostros en la imagen de prueba...")

faces = face_client.detect_from_url(
    url=test_image,
    detection_model=FaceDetectionModel.DETECTION03,
    recognition_model=FaceRecognitionModel.RECOGNITION04,
    return_face_id=True,
    return_face_attributes=[FaceAttributeTypeRecognition04.QUALITY_FOR_RECOGNITION],
)

print(f"   Total de rostros detectados: {len(faces)}")

face_ids = []
for face in faces:
    quality = face.face_attributes.quality_for_recognition
    if quality != QualityForRecognition.LOW:
        face_ids.append(face.face_id)
        print(f"   ✅ Face ID: {face.face_id[:8]}... (calidad: {quality})")
    else:
        print(f"   ❌ Face ID: {face.face_id[:8]}... (calidad: {quality} - excluido)")

print(f"\n📊 Rostros válidos para identificación: {len(face_ids)}")

## 13. Identificación de Rostros

In [ ]:
print("🔎 Identificando rostros contra el Large Person Group...")
print("=" * 60)

identify_results = face_client.identify_from_large_person_group(
    face_ids=face_ids,
    large_person_group_id=LARGE_PERSON_GROUP_ID,
)

for identify_result in identify_results:
    face_id_short = identify_result.face_id[:8]
    
    if identify_result.candidates:
        top_candidate = identify_result.candidates[0]
        confidence = top_candidate.confidence
        person_id = top_candidate.person_id
        
        indicator = "🟢" if confidence > 0.9 else "🟡" if confidence > 0.7 else "🔴"
        
        print(f"\n{indicator} Face {face_id_short}... IDENTIFICADO")
        print(f"   Person ID: {person_id}")
        print(f"   Confianza: {confidence:.2%}")
    else:
        print(f"\n❓ Face {face_id_short}... NO IDENTIFICADO")

print("\n" + "=" * 60)

## 14. Verificación de Identidad

In [ ]:
print("✔️ Verificando identidades...")
print("=" * 60)

for identify_result in identify_results:
    if identify_result.candidates:
        face_id = identify_result.face_id
        person_id = identify_result.candidates[0].person_id
        
        verify_result = face_client.verify_from_large_person_group(
            face_id=face_id,
            large_person_group_id=LARGE_PERSON_GROUP_ID,
            person_id=person_id,
        )
        
        status = "✅ VERIFICADO" if verify_result.is_identical else "❌ NO VERIFICADO"
        print(f"\nFace {face_id[:8]}... vs Person {str(person_id)[:8]}...")
        print(f"   Resultado: {status}")
        print(f"   Confianza: {verify_result.confidence:.2%}")

print("\n" + "=" * 60)

## 15. Limpieza de Recursos

In [ ]:
print("🗑️ Eliminando Large Person Group...")

face_admin_client.large_person_group.delete(LARGE_PERSON_GROUP_ID)

print(f"✅ El grupo {LARGE_PERSON_GROUP_ID} ha sido eliminado")

## 16. Cierre de Clientes

In [ ]:
face_admin_client.close()
face_client.close()

print("✅ Clientes cerrados correctamente")

---

## Resumen

En este notebook hemos demostrado el flujo completo de trabajo con Azure Face API.

### Referencias

- [Documentación oficial de Azure Face API](https://learn.microsoft.com/azure/cognitive-services/face/)
- [SDK azure-ai-vision-face en PyPI](https://pypi.org/project/azure-ai-vision-face/)

---

**Fin del Quickstart** 🎉